In [28]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
import os

print(os.listdir("/content/drive/MyDrive"))

['Name_ Ansari Na-WPS Office.pdf', 'ct form  document (3).pdf', 'ct form  document (2).pdf', 'ct form  document (1).pdf', 'ct form  document.pdf', 'Classroom', 'property Document.pdf', 'Reflection', 'Untitled form (1).gform', 'ppt ', 'ppt', 'pptttt', 'Untitled document (7).gdoc', 'Saif .pdf', 'fees.pdf', 'Untitled document (6).gdoc', 'Mu.pdf', 'exam form.pdf', 'scholarship form.pdf', 'CP Lab Pr 1 - RD.pdf', 'CP Lab Pr 2 - RD.pdf', 'Share Unit-3 Transformer.pptx', 'Share Chapter-5 Diode PPT.pptx', 'Share DC Motor Theory.pdf', 'Blank.gform', 'Share hall ticket .pdf', 'Share saif.pdf', 'passbook.pdf', 'experiment1a.py', 'experiment1b.py', '24AI18.pdf', '24AI18(1).pdf', 'Untitled form.gform', 'Registration .gform', 'pract1.1.py', 'Untitled document (5).gdoc', 'Activity-3.gform', 'Untitled document (4).gdoc', 'Untitled document (3).gdoc', 'EXPERIMENT NO:01.gdoc', 'photo.pdf', 'Colab Notebooks', 'Untitled spreadsheet.gsheet', 'Untitled document (2).gdoc', 'ML EXP 1.gdoc', "I am sharing 'expe

In [30]:
import os

BASE_DIR = "/content/drive/MyDrive/deepfake-defense/datasets"

TRAIN_DIR = os.path.join(BASE_DIR, "processed")
TEST_DIR = os.path.join(BASE_DIR, "processed_test")

# Create directories if they don't exist
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

print("TRAIN:", TRAIN_DIR)
print("TEST :", TEST_DIR)

print("\nTrain folders:")
print(os.listdir(TRAIN_DIR))

print("\nTest folders:")
print(os.listdir(TEST_DIR))

TRAIN: /content/drive/MyDrive/deepfake-defense/datasets/processed
TEST : /content/drive/MyDrive/deepfake-defense/datasets/processed_test

Train folders:
[]

Test folders:
[]


In [31]:
!pip install -q timm

In [32]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from glob import glob
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms

import timm

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

In [33]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [34]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cpu


In [35]:
real_images = (
    glob(os.path.join(TRAIN_DIR, "real", "*.jpg")) +
    glob(os.path.join(TRAIN_DIR, "real", "*.jpeg")) +
    glob(os.path.join(TRAIN_DIR, "real", "*.png"))
)

fake_images = (
    glob(os.path.join(TRAIN_DIR, "fake", "*.jpg")) +
    glob(os.path.join(TRAIN_DIR, "fake", "*.jpeg")) +
    glob(os.path.join(TRAIN_DIR, "fake", "*.png"))
)

print("Real images:", len(real_images))
print("Fake images:", len(fake_images))
print("Total:", len(real_images) + len(fake_images))

Real images: 0
Fake images: 0
Total: 0


In [36]:
all_images = real_images + fake_images

all_labels = (
    [0] * len(real_images) +
    [1] * len(fake_images)
)

print(Counter(all_labels))

Counter()


In [38]:
import os

DATA_ROOT = "/content/drive/MyDrive"

print("Searching for FaceForensics++ folders...\n")

for root, dirs, files in os.walk(DATA_ROOT):
    folder_name = os.path.basename(root).lower()

    if any(x in folder_name for x in [
        "faceforensics",
        "faceforensics++",
        "ff++",
        "deepfakes",
        "original",
        "manipulated"
    ]):
        print(root)

Searching for FaceForensics++ folders...



In [39]:
import os

ROOT = "/content/drive/MyDrive"

print("Top-level folders in MyDrive:\n")

for item in os.listdir(ROOT):
    full_path = os.path.join(ROOT, item)

    if os.path.isdir(full_path):
        print("📁", item)

Top-level folders in MyDrive:

📁 Classroom
📁 Reflection
📁 Colab Notebooks
📁 deepfake-defense


In [40]:
import os

PROJECT_DIR = "/content/drive/MyDrive/deepfake-defense"

print("Contents of deepfake-defense:\n")

for item in os.listdir(PROJECT_DIR):
    full_path = os.path.join(PROJECT_DIR, item)

    if os.path.isdir(full_path):
        print("📁", item)
    else:
        print("📄", item)

Contents of deepfake-defense:

📁 datasets


In [41]:
DATASETS_DIR = "/content/drive/MyDrive/deepfake-defense/datasets"

print("Contents of datasets:\n")

for item in os.listdir(DATASETS_DIR):
    full_path = os.path.join(DATASETS_DIR, item)

    if os.path.isdir(full_path):
        print("📁", item)
    else:
        print("📄", item)

Contents of datasets:

📁 processed
📁 processed_test


In [42]:
TRAIN_DIR = "/content/drive/MyDrive/deepfake-defense/datasets/processed"
TEST_DIR = "/content/drive/MyDrive/deepfake-defense/datasets/processed_test"

print("PROCESSED:")
for item in os.listdir(TRAIN_DIR):
    full_path = os.path.join(TRAIN_DIR, item)

    if os.path.isdir(full_path):
        print("📁", item)
    else:
        print("📄", item)

print("\nPROCESSED_TEST:")
for item in os.listdir(TEST_DIR):
    full_path = os.path.join(TEST_DIR, item)

    if os.path.isdir(full_path):
        print("📁", item)
    else:
        print("📄", item)

PROCESSED:

PROCESSED_TEST:


In [43]:
import os

DATASETS_DIR = "/content/drive/MyDrive/deepfake-defense/datasets"

print("Searching everything inside datasets...\n")

for root, dirs, files in os.walk(DATASETS_DIR):
    print("FOLDER:", root)

    if dirs:
        print("  Subfolders:", dirs)

    if files:
        print("  Files:", files[:10], "..." if len(files) > 10 else "")

    print()

Searching everything inside datasets...

FOLDER: /content/drive/MyDrive/deepfake-defense/datasets
  Subfolders: ['processed', 'processed_test']

FOLDER: /content/drive/MyDrive/deepfake-defense/datasets/processed

FOLDER: /content/drive/MyDrive/deepfake-defense/datasets/processed_test



In [44]:
import os

print("Searching MyDrive for video files...\n")

video_extensions = (".mp4", ".avi", ".mov", ".mkv", ".webm")

found = 0

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    videos = [f for f in files if f.lower().endswith(video_extensions)]

    if videos:
        print("FOLDER:", root)
        print("Videos:", len(videos))
        print("Examples:", videos[:5])
        print("-" * 60)
        found += len(videos)

print("\nTOTAL VIDEOS FOUND:", found)

Searching MyDrive for video files...


TOTAL VIDEOS FOUND: 0


In [45]:
!python download-FaceForensics.py -h

usage: download-FaceForensics.py [-h]
                                 [-d {original_youtube_videos,original_youtube_videos_info,original,DeepFakeDetection_original,Deepfakes,DeepFakeDetection,Face2Face,FaceShifter,FaceSwap,NeuralTextures,all}]
                                 [-c {raw,c23,c40}] [-t {videos,masks,models}]
                                 [-n NUM_VIDEOS] [--server {EU,EU2,CA}]
                                 output_path

Downloads FaceForensics v2 public data release.

positional arguments:
  output_path           Output directory.

options:
  -h, --help            show this help message and exit
  -d, --dataset {original_youtube_videos,original_youtube_videos_info,original,DeepFakeDetection_original,Deepfakes,DeepFakeDetection,Face2Face,FaceShifter,FaceSwap,NeuralTextures,all}
                        Which dataset to download, either pristine or
                        manipulated data or the downloaded youtube videos.
                        (default: all)
  -c, --co

In [ ]:
!python download-FaceForensics.py \
    /content/drive/MyDrive/deepfake-defense/datasets/FF++ \
    -d original \
    -c c23 \
    -t videos \
    -n 50

By pressing any key to continue you confirm that you have agreed to the FaceForensics terms of use as described at:
http://canis.vc.in.tum.de:8100/webpage/FaceForensics_TOS.pdf
***
Press any key to continue, or CTRL-C to exit.
Traceback (most recent call last):
  File "/content/download-FaceForensics.py", line 261, in <module>
    main(args)
  File "/content/download-FaceForensics.py", line 144, in main
    _ = input('')
KeyboardInterrupt
^C


In [ ]:
!python download-FaceForensics.py \
    /content/drive/MyDrive/deepfake-defense/datasets/FF++ \
    -d Deepfakes \
    -c c23 \
    -t videos \
    -n 50

By pressing any key to continue you confirm that you have agreed to the FaceForensics terms of use as described at:
http://canis.vc.in.tum.de:8100/webpage/FaceForensics_TOS.pdf
***
Press any key to continue, or CTRL-C to exit.


In [ ]:
import os

FF_DIR = "/content/drive/MyDrive/deepfake-defense/datasets/FF++"

for root, dirs, files in os.walk(FF_DIR):
    video_files = [
        f for f in files
        if f.lower().endswith((".mp4", ".avi", ".mov", ".mkv"))
    ]

    if video_files:
        print("FOLDER:", root)
        print("Videos:", len(video_files))
        print("Examples:", video_files[:3])

In [46]:
!ls

download-FaceForensics.py  drive  sample_data


In [ ]:
train_images, val_images, train_labels, val_labels = train_test_split(
    all_images,
    all_labels,
    test_size=0.20,
    random_state=SEED,
    stratify=all_labels
)

print("Training images:", len(train_images))
print("Validation images:", len(val_images))

print("\nTraining distribution:")
print(Counter(train_labels))

print("\nValidation distribution:")
print(Counter(val_labels))

In [ ]:
import os

print("TRAIN_DIR:")
print(TRAIN_DIR)

print("\nContents:")
for item in os.listdir(TRAIN_DIR):
    print(repr(item))

In [ ]:
import os

image_extensions = (
    ".jpg", ".jpeg", ".png",
    ".JPG", ".JPEG", ".PNG"
)

all_found_images = []

for root, dirs, files in os.walk(TRAIN_DIR):
    for file in files:
        if file.endswith(image_extensions):
            all_found_images.append(
                os.path.join(root, file)
            )

print("Total images found:", len(all_found_images))

print("\nFirst 10 images:")
for path in all_found_images[:10]:
    print(path)

In [ ]:
for root, dirs, files in os.walk(TRAIN_DIR):
    print("\nFOLDER:", root)
    print("Subfolders:", dirs)
    print("Number of files:", len(files))

In [ ]:
import os

IMAGE_EXTENSIONS = (
    ".jpg", ".jpeg", ".png",
    ".JPG", ".JPEG", ".PNG"
)

real_images = []
fake_images = []

for root, dirs, files in os.walk(TRAIN_DIR):

    for file in files:

        if not file.endswith(IMAGE_EXTENSIONS):
            continue

        full_path = os.path.join(root, file)

        folder_name = os.path.basename(root).lower()

        if folder_name == "real":
            real_images.append(full_path)

        elif folder_name == "fake":
            fake_images.append(full_path)


print("Real images:", len(real_images))
print("Fake images:", len(fake_images))
print("Total images:", len(real_images) + len(fake_images))

In [ ]:
all_images = real_images + fake_images

all_labels = (
    [0] * len(real_images) +
    [1] * len(fake_images)
)

print("Total:", len(all_images))
print("Labels:", len(all_labels))

In [ ]:
from sklearn.model_selection import train_test_split

train_images, val_images, train_labels, val_labels = train_test_split(
    all_images,
    all_labels,
    test_size=0.20,
    random_state=SEED,
    stratify=all_labels
)

print("Training images:", len(train_images))
print("Validation images:", len(val_images))

In [ ]:
IMG_SIZE = 224

In [ ]:
train_transform = transforms.Compose([

    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(5),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
val_transform = transforms.Compose([

    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
class DeepfakeDataset(Dataset):

    def __init__(
        self,
        image_paths,
        labels,
        transform=None
    ):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        image_path = self.image_paths[idx]

        image = Image.open(
            image_path
        ).convert("RGB")

        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(
            label,
            dtype=torch.long
        )

In [ ]:
train_dataset = DeepfakeDataset(
    train_images,
    train_labels,
    transform=train_transform
)

val_dataset = DeepfakeDataset(
    val_images,
    val_labels,
    transform=val_transform
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

In [ ]:
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

In [ ]:
images, labels = next(iter(train_loader))

plt.figure(figsize=(12, 8))

for i in range(8):

    plt.subplot(2, 4, i + 1)

    image = images[i].permute(1, 2, 0).numpy()

    # Undo normalization
    image = image * np.array(
        [0.229, 0.224, 0.225]
    ) + np.array(
        [0.485, 0.456, 0.406]
    )

    image = np.clip(image, 0, 1)

    plt.imshow(image)

    label = "Fake" if labels[i].item() == 1 else "Real"

    plt.title(label)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
xception = timm.create_model(
    "xception",
    pretrained=True,
    num_classes=0
)

In [ ]:
dummy = torch.randn(1, 3, 224, 224)

with torch.no_grad():
    output = xception(dummy)

print(output.shape)

In [ ]:
xception_dim = output.shape[1]

print("Xception feature dimension:", xception_dim)

In [ ]:
vit = timm.create_model(
    "vit_base_patch16_224",
    pretrained=True,
    num_classes=0
)

In [ ]:
with torch.no_grad():
    output = vit(dummy)

print(output.shape)

In [ ]:
vit_dim = output.shape[1]

print("ViT feature dimension:", vit_dim)

In [ ]:
class XceptionViT(nn.Module):

    def __init__(self):

        super().__init__()

        # -------------------------
        # Xception branch
        # -------------------------

        self.xception = timm.create_model(
            "xception",
            pretrained=True,
            num_classes=0
        )

        # -------------------------
        # ViT branch
        # -------------------------

        self.vit = timm.create_model(
            "vit_base_patch16_224",
            pretrained=True,
            num_classes=0
        )

        # Determine feature dimensions

        self.xception_dim = self.xception.num_features
        self.vit_dim = self.vit.num_features

        print(
            "Xception features:",
            self.xception_dim
        )

        print(
            "ViT features:",
            self.vit_dim
        )

        # -------------------------
        # Fusion classifier
        # -------------------------

        fusion_dim = (
            self.xception_dim +
            self.vit_dim
        )

        self.classifier = nn.Sequential(

            nn.Linear(
                fusion_dim,
                512
            ),

            nn.BatchNorm1d(512),

            nn.ReLU(),

            nn.Dropout(0.4),

            nn.Linear(
                512,
                128
            ),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                128,
                2
            )
        )

    def forward(self, x):

        # Xception features
        xception_features = self.xception(x)

        # ViT features
        vit_features = self.vit(x)

        # Concatenate
        combined = torch.cat(
            [
                xception_features,
                vit_features
            ],
            dim=1
        )

        # Classification
        output = self.classifier(
            combined
        )

        return output

In [ ]:
model = XceptionViT()

model = model.to(device)

In [ ]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    f"Total parameters: "
    f"{total_params:,}"
)

print(
    f"Trainable parameters: "
    f"{trainable_params:,}"
)

In [ ]:
for param in model.xception.parameters():
    param.requires_grad = False

for param in model.vit.parameters():
    param.requires_grad = False

In [ ]:
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    f"Trainable parameters: "
    f"{trainable_params:,}"
)

In [ ]:
print(Counter(train_labels))

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
class_counts = Counter(train_labels)

total = len(train_labels)

class_weights = torch.tensor(
    [
        total / class_counts[0],
        total / class_counts[1]
    ],
    dtype=torch.float32
)

class_weights = (
    class_weights /
    class_weights.sum()
)

class_weights = class_weights.to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

print("Class weights:", class_weights)

In [ ]:
optimizer = optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=1e-4,
    weight_decay=1e-4
)

In [ ]:
optimizer = optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=1e-4,
    weight_decay=1e-4
)

In [ ]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

In [ ]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer
):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += (
            loss.item() *
            labels.size(0)
        )

        predictions = outputs.argmax(
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    loss = running_loss / total

    accuracy = correct / total

    return loss, accuracy

In [ ]:
def validate(
    model,
    loader,
    criterion
):

    model.eval()

    running_loss = 0.0

    all_labels = []
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            running_loss += (
                loss.item() *
                labels.size(0)
            )

            probabilities = torch.softmax(
                outputs,
                dim=1
            )

            predictions = outputs.argmax(
                dim=1
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_probabilities.extend(
                probabilities[:, 1]
                .cpu()
                .numpy()
            )

    loss = (
        running_loss /
        len(loader.dataset)
    )

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    return (
        loss,
        accuracy,
        all_labels,
        all_predictions,
        all_probabilities
    )

In [ ]:
EPOCHS = 15

best_val_loss = float("inf")

train_losses = []
val_losses = []

train_accs = []
val_accs = []

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer
    )

    (
        val_loss,
        val_acc,
        val_labels,
        val_predictions,
        val_probs
    ) = validate(
        model,
        val_loader,
        criterion
    )

    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            "/content/xception_vit_fusion_best.pth"
        )

        print("✓ Best model saved")

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    train_losses,
    label="Training Loss"
)

plt.plot(
    val_losses,
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title(
    "Xception + ViT Training Loss"
)

plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    train_accs,
    label="Training Accuracy"
)

plt.plot(
    val_accs,
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.title(
    "Xception + ViT Accuracy"
)

plt.legend()

plt.show()

In [ ]:
model.load_state_dict(
    torch.load(
        "/content/xception_vit_fusion_best.pth",
        map_location=device
    )
)

model.to(device)

print("Best model loaded.")

In [ ]:
(
    val_loss,
    val_acc,
    y_true,
    y_pred,
    y_prob
) = validate(
    model,
    val_loader,
    criterion
)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

auc = roc_auc_score(
    y_true,
    y_prob
)

print("\nXCEPTION + VIT RESULTS")
print("======================")

print(f"Accuracy : {val_acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")

In [ ]:
print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Real",
            "Fake"
        ],
        digits=4
    )
)

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Real", "Fake"],
    yticklabels=["Real", "Fake"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.title(
    "Xception + ViT Confusion Matrix"
)

plt.show()

In [ ]:
fpr, tpr, thresholds = roc_curve(
    y_true,
    y_prob
)

plt.figure(figsize=(7, 5))

plt.plot(
    fpr,
    tpr,
    label=f"AUC = {auc:.4f}"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "Xception + ViT ROC Curve"
)

plt.legend()

plt.show()

In [ ]:
for param in model.xception.parameters():
    param.requires_grad = False

for param in model.vit.parameters():
    param.requires_grad = False

In [ ]:
for name, param in model.xception.named_parameters():
    print(name)

In [ ]:
for name, param in model.vit.named_parameters():
    print(name)

In [ ]:
lr = 1e-5

In [ ]:
FINAL_MODEL_PATH = os.path.join(
    BASE_DIR,
    "xception_vit_deepfake_best.pth"
)

torch.save(
    model.state_dict(),
    FINAL_MODEL_PATH
)

print(
    "Model saved to:",
    FINAL_MODEL_PATH
)

In [ ]:
def predict_image(image_path):

    model.eval()

    image = Image.open(
        image_path
    ).convert("RGB")

    image_tensor = val_transform(
        image
    ).unsqueeze(0).to(device)

    with torch.no_grad():

        output = model(
            image_tensor
        )

        probability = torch.softmax(
            output,
            dim=1
        )[0]

    real_probability = (
        probability[0].item()
    )

    fake_probability = (
        probability[1].item()
    )

    if fake_probability >= 0.5:
        prediction = "DEEPFAKE"
    else:
        prediction = "REAL"

    return (
        prediction,
        real_probability,
        fake_probability
    )

In [ ]:
sample_image = val_images[0]

prediction, real_prob, fake_prob = (
    predict_image(sample_image)
)

print("Prediction:", prediction)
print(
    f"Real: {real_prob:.2%}"
)

print(
    f"Fake: {fake_prob:.2%}"
)

In [ ]:
def show_prediction(image_path):

    prediction, real_prob, fake_prob = (
        predict_image(image_path)
    )

    image = Image.open(
        image_path
    )

    plt.figure(figsize=(6, 6))

    plt.imshow(image)

    plt.title(
        f"{prediction}\n"
        f"Real: {real_prob:.2%} | "
        f"Fake: {fake_prob:.2%}"
    )

    plt.axis("off")

    plt.show()

In [ ]:
show_prediction(
    val_images[0]
)